# Entendimento da atividade

Inicialmente, existem calculadoras tradicionais para determinar indicadores como TIR, VPL e fluxo de caixa de forma técnica. Contudo, para facilitar a usabilidade, foi desenvolvido um código que permite visualizar gráficos e inserir diferentes inputs, oferecendo maior flexibilidade ao usuário.

A aplicação apresenta o fluxo de caixa ao longo dos anos com base no investimento realizado. Para isso, o usuário pode inserir valores de fluxo de caixa, taxa de desconto e investimento inicial.

Com essas informações, o sistema calcula automaticamente o Payback, a TIR e o VPL, além de permitir a visualização interativa do gráfico do fluxo de caixa da empresa ao longo do período analisado.

Dessa forma, a ferramenta auxilia na interpretação dos dados e indica se o investimento é viável para a gestão.

In [95]:
import pandas as pd
import numpy_financial as npf

def build_cashflow_dataframe(years, cashflows):
    return pd.DataFrame({
        'Ano': years,
        'Fluxo de Caixa': cashflows
    })

def calculate_npv(rate, cashflows):
    # npf.npv expects the rate as a decimal and initial investment already included
    return npf.npv(rate, cashflows)

def calculate_irr(cashflows):
    try:
        irr = npf.irr(cashflows)
        message = ""
        return irr, message
    except Exception as e:
        return float('nan'), f"Não foi possível calcular a TIR: {e}"

def format_currency(value):
    return f"R$ {value:,.2f}".replace(',', 'X').replace('.', ',').replace('X', '.')

def format_percentage(value):
    if np.isnan(value):
        return "N/A"
    return f"{value * 100:,.2f}%"

discount_rate = 0.12
initial_investment = 91616
operational_cashflows = [19768, 20870, 29186, 36291, 45254]

final_years = ["Ano 0", 2026, 2027, 2028, 2029, 2030]
final_cashflows = [-initial_investment] + operational_cashflows

final_df = build_cashflow_dataframe(final_years, final_cashflows)

npv_value = calculate_npv(discount_rate, final_cashflows)
irr_value, irr_message = calculate_irr(final_cashflows)

npv_future_only = sum(
    cf / ((1 + discount_rate) ** t)
    for t, cf in enumerate(operational_cashflows, start=1)
)

results_df = pd.DataFrame({
    "indicator": [
        "VPL dos fluxos futuros",
        "VPL líquido do projeto",
        "TIR"
    ],
    "result_in_python": [
        format_currency(npv_future_only),
        format_currency(npv_value),
        format_percentage(irr_value)
    ]
})

print("Série de fluxos usada no cálculo:")
print(final_cashflows)
print()

print(final_df)
print()

print("Resultados em Python")
print(f"Investimento inicial: {format_currency(initial_investment)}")
print(f"Taxa de desconto: {format_percentage(discount_rate)}")
print(f"VPL dos fluxos futuros: {format_currency(npv_future_only)}")
print(f"VPL líquido do projeto: {format_currency(npv_value)}")
print(f"TIR: {format_percentage(irr_value)}")
print(f"Mensagem da TIR: {irr_message}")

display(results_df)

Série de fluxos usada no cálculo:
[-91616, 19768, 20870, 29186, 36291, 45254]

     Ano  Fluxo de Caixa
0  Ano 0          -91616
1   2026           19768
2   2027           20870
3   2028           29186
4   2029           36291
5   2030           45254

Resultados em Python
Investimento inicial: R$ 91.616,00
Taxa de desconto: 12.00%
VPL dos fluxos futuros: R$ 103.803,38
VPL líquido do projeto: R$ 12.187,38
TIR: 16.49%
Mensagem da TIR: 


,indicator,result_in_python
0,VPL dos fluxos futuros,"R$ 103.803,38"
1,VPL líquido do projeto,"R$ 12.187,38"
2,TIR,16.49%


In [80]:
!pip install gradio numpy_financial -q

In [86]:
import gradio as gr
import numpy as np
import numpy_financial as npf
import matplotlib.pyplot as plt
import plotly.graph_objects as go

def calcular_vpl(fluxos, taxa):
  """Calcula o Valor Presente Líquido (VPL)."""

  return npf.npv(taxa / 100, fluxos)

def calcular_tir(fluxos):
  """Calcula a Taxa Interna de Retorno (TIR)."""
  # npf.irr expects the full series of cash flows
  return npf.irr(fluxos)

def calcular_payback_descontado(fluxos, taxa):
  """Calcula o Payback Descontado."""
  taxa_decimal = taxa / 100
  investimento_inicial_abs = abs(fluxos[0]) # Use absolute value for internal calculation

  saldo_acumulado_descontado = -investimento_inicial_abs # Start with negative of absolute initial investment

  if investimento_inicial_abs <= 0: # Handle cases where initial investment is not negative (should not happen with abs)
    return 0.0

  for i in range(1, len(fluxos)):
      fluxo_descontado = fluxos[i] / ((1 + taxa_decimal)**i)
      saldo_acumulado_descontado += fluxo_descontado

      if saldo_acumulado_descontado >= 0:
          # Payback occurs in this period
          periodo_anterior_saldo = saldo_acumulado_descontado - fluxo_descontado
          # Calculate fractional part of the year
          if fluxo_descontado != 0:
            return (i - 1) + (abs(periodo_anterior_saldo) / fluxo_descontado)
          else:
            return i # if discounted cash flow is zero, payback is at end of this period

  return float('inf') # Payback never occurs within the given periods

def plotar_barras_fluxos(fluxos):
  """Gera um gráfico de barras dos fluxos de caixa com Plotly para interatividade."""
  anos = list(range(len(fluxos)))
  cores = ['red' if f < 0 else 'green' for f in fluxos]

  fig = go.Figure(data=[go.Bar(
      x=anos,
      y=fluxos,
      marker_color=cores,
      # Custom text for hover info
      text=[f'Período: {a}<br>Fluxo: R$ {f:,.2f}' for a, f in zip(anos, fluxos)],
      hoverinfo='text' # Display only custom text on hover
  )])

  fig.update_layout(
      title="Fluxo de Caixa ao Longo do Tempo",
      xaxis_title="Período (0 = Investimento Inicial)",
      yaxis_title="Fluxo de Caixa (R$)",
      xaxis=dict(showgrid=True, gridwidth=1, gridcolor='LightGray'),
      yaxis=dict(showgrid=True, gridwidth=1, gridcolor='LightGray')
  )
  return fig

def analisar_investimento(investimento_inicial_input, fluxos_futuros_texto, taxa):
  try:
    # Ensure initial investment is always negative
    investimento_inicial = -abs(investimento_inicial_input)

    # Parse future cash flows, handling thousands separator ('.')
    if fluxos_futuros_texto.strip():
        # Replace '.' (thousands separator) with '' before converting to float
        fluxos_futuros = [float(x.strip().replace('.', '')) for x in fluxos_futuros_texto.split(',')]
    else:
        fluxos_futuros = []

    # Combine initial investment with future cash flows
    fluxos = [investimento_inicial] + fluxos_futuros

    if len(fluxos) < 2: # Need at least initial investment and one future cash flow
      return "Erro: Mínimo 1 fluxo futuro necessário (além do investimento inicial)", "", None

    # Convert tax to decimal for financial functions, but keep as percentage for display if needed
    # Note: npf.npv expects the investment at time 0 (fluxos[0]) and subsequent flows
    vpl = calcular_vpl(fluxos, float(taxa))
    tir = calcular_tir(fluxos)
    payback = calcular_payback_descontado(fluxos, float(taxa))

    resultado = f"VPL: R$ {vpl:,.2f}\nTIR: {tir*100:.2f}% a.a.\nPayback: {payback:.2f} anos"
    decisao = "✓ VIÁVEL" if vpl > 0 else "✗ INVIÁVEL"

    return resultado, decisao, plotar_barras_fluxos(fluxos)
  except Exception as e:
    return f"Erro: {e}", "", None

# Prepare the cash flow data from the user's input
# Original user_fluxos_texto was "-91.616, 19.768, 20.870, 29.186, 36.291, 45.254"
user_investimento_inicial = 91.616 # User will input a positive value for initial investment
user_fluxos_futuros_texto = "19.768, 20.870, 29.186, 36.291, 45.254"
# Use the taxa_minima from kernel state, convert to percentage for Gradio input
user_taxa_desconto_percent = taxa_minima * 100

demo = gr.Interface(
  fn=analisar_investimento,
  inputs=[
    gr.Number(label="Investimento Inicial (R$)", value=user_investimento_inicial),
    gr.Textbox(label="Fluxos de Caixa Futuros (separados por vírgula)",
               value=user_fluxos_futuros_texto),
    gr.Number(label="Taxa de Desconto (%)", value=user_taxa_desconto_percent)
  ],
  outputs=[
    gr.Textbox(label="Resultados"),
    gr.Textbox(label="Decisão"),
    gr.Plot(label="Visualização")
  ],
  title="Análise de Investimento - VPL/TIR"
)
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://52e36dc3e28cbaadb7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Gráficos abaixo

Abaixo conseguimos ver a sensibilidade do VPL de acordo à taxa de desconto e também o fluxo de caixa pelo período analisado.

In [87]:
import plotly.graph_objects as go
import numpy as np

def plotar_barras_fluxos(fluxos_caixa: list[float]):
  periodos = list(range(len(fluxos_caixa)))
  cores = ['#ff4545' if f < 0 else '#89cea5' for f in fluxos_caixa]

  fig = go.Figure()
  fig.add_trace(go.Bar(
    x=periodos,
    y=fluxos_caixa,
    marker_color=cores,
    text=[f'R$ {f:,.0f}' for f in fluxos_caixa],
    textposition='outside'
  ))
  fig.update_layout(
    title='Fluxos de Caixa por Período',
    xaxis_title='Período (anos)',
    yaxis_title='Fluxo de Caixa (R$)',
    showlegend=False
  )
  return fig

def plotar_sensibilidade_vpl(fluxos_caixa: list[float], taxa_min: float = 0, taxa_max: float = 0.30):
  # Convert percentage input for calcular_vpl to decimal for calculation
  # The original calcular_vpl expects percentage (e.g., 12 for 12%), so we need to adjust 'taxas' values
  taxas_decimais = [i/100 for i in range(int(taxa_min*100), int(taxa_max*100)+1, 1)]
  vpls = [calcular_vpl(fluxos_caixa, t*100) for t in taxas_decimais] # Pass as percentage to calcular_vpl

  fig = go.Figure()
  fig.add_trace(go.Scatter(
    x=[t*100 for t in taxas_decimais], # Display as percentage on x-axis
    y=vpls,
    mode='lines',
    line=dict(color='#2e2640', width=3)
  ))
  fig.add_hline(y=0, line_dash="dash", line_color="red")
  fig.update_layout(
    title='Sensibilidade do VPL à Taxa de Desconto',
    xaxis_title='Taxa de Desconto (%)',
    yaxis_title='VPL (R$)',
    hovermode='x unified'
  )
  return fig

# fluxo_completo is already a list, no need to call .tolist()
fluxos_completos = fluxo_completo

fig1 = plotar_barras_fluxos(fluxos_completos)
fig1.show()

fig2 = plotar_sensibilidade_vpl(fluxos_completos)
fig2.show()